# Make modality disagreement visible

Prerequisites: Class scores, label alignment and calibration.

Two input types can disagree even when each produces a confident output. The fixture combines text scores (0.9,0.1) and image scores (0.1,0.9) using a chosen image weight. At equal weight the fused scores are (0.5,0.5), so the displayed first-class choice is merely the implementation's tie break.

Linear fusion assumes compatible labels and interpretable score scales. Neither is guaranteed by putting two models next to each other. If the image is absent, this fixture explicitly falls back to text; replacing missing data with a confident all-zero vector would have a different meaning. Real evaluation should include contradiction, missingness and subgroup slices, not just aggregate accuracy.

These are fictional score vectors, not image/text embeddings or an implementation of CLIP. The referenced work motivates studying alignment; it does not validate this hand-selected fusion policy.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Text | 0.9,0.1 | First-class evidence |
| Image | 0.1,0.9 | Contradictory second-class evidence |
| Fusion | Weighted average or explicit fallback | Preserve disagreement |

## Predict

Weight 0.5 yields a tie; weight 0.8 favors class 1. Missing image returns the original text vector rather than pretending an image was processed.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import extensions as m

In [ ]:
result=m.multimodal()
assert result['fused']==[.5,.5]
assert m.multimodal(.8)['decision']==1
assert m.multimodal(missing=True)['fused']==[.9,.1]
print(result)

## Perturb and explain

Introduce a third class and a mismatched label order. Write a schema check that rejects the mismatch before fusing scores.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

An equal-weight tie demonstrates:

1. Certain agreement
2. Unresolved opposing evidence under this policy
3. That one modality is necessarily wrong

Answer and explanations: 1: The input vectors disagree. 2: The arithmetic alone does not resolve the conflict. 3: The fixture has no independent ground-truth observation.

## Primary evidence

- [Radford et al. — Learning Transferable Visual Models From Natural Language Supervision](https://proceedings.mlr.press/v139/radford21a.html)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.